<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 1. Google Drive 연결 </h2>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 2. 데이터셋 ZIP 파일 경로 설정 </h2>

In [ ]:
from pathlib import Path

ZIP_CANDIDATES = [
    Path('/content/drive/MyDrive/wardy/ml/src/data/person_dataset/person_dataset_merged.zip'),
    Path('/content/drive/MyDrive/AI_Project/wardy/ml/src/data/person_dataset/person_dataset_merged.zip'),
]

zip_matches = [path for path in ZIP_CANDIDATES if path.is_file()]
if not zip_matches:
    zip_matches = list(Path('/content/drive/MyDrive').glob('**/person_dataset_merged.zip'))

if not zip_matches:
    raise FileNotFoundError(
        'Google Drive에서 person_dataset_merged.zip을 찾을 수 없습니다. '
        'Drive 업로드 위치와 로그인 계정을 확인하세요.'
    )

ZIP_PATH = str(zip_matches[0])
LOCAL_ZIP = '/content/person_dataset_merged.zip'
print('사용할 데이터셋 ZIP:', ZIP_PATH)

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 3. ZIP 파일 Colab 로컬 환경으로 복사 </h2>

In [ ]:
import shutil

shutil.copy2(ZIP_PATH, LOCAL_ZIP)

print("ZIP 파일 복사 완료")

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 4. 데이터셋 ZIP 파일 압축 해제 </h2>

In [ ]:
import zipfile

with zipfile.ZipFile(LOCAL_ZIP, "r") as zip_ref:
    zip_ref.extractall("/content")

print("압축 해제 완료")

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 5. 데이터셋 경로 설정 </h2>

In [ ]:
DATASET_PATH = "/content/person_dataset"
DATA_YAML = f"{DATASET_PATH}/data.yaml"

print(DATASET_PATH)
print(DATA_YAML)

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 6. data.yaml 경로 수정 </h2>

In [ ]:
from pathlib import Path

yaml_path = Path(DATA_YAML)

text = yaml_path.read_text(encoding="utf-8")
text = text.replace("path: .", "path: /content/person_dataset")

yaml_path.write_text(text, encoding="utf-8")

print(yaml_path.read_text(encoding="utf-8"))

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 7. Ultralytics YOLO 설치 </h2>

In [ ]:
!pip install -q -U ultralytics

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 8. GPU 사용 가능 여부 확인 </h2>

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'CUDA GPU가 할당되지 않았습니다. Colab에서는 런타임 유형을 T4 GPU로 변경하고, '
        'VS Code에서는 NVIDIA GPU가 연결된 Jupyter 커널을 선택하세요.'
    )

DEVICE = 0
print('CUDA 사용 가능:', True)
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(DEVICE))
print('GPU 개수:', torch.cuda.device_count())

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 9. 5 Epoch 시험 학습 </h2>

In [ ]:
from ultralytics import YOLO
from pathlib import Path

RUN_DIR = Path('/content/drive/MyDrive/wardy/ml/src/export/person_detector_smoke_test')
model = YOLO('yolo11n.pt')
results = model.train(
    data=DATA_YAML,
    epochs=5,
    imgsz=640,
    batch=8,
    device=DEVICE,
    workers=2,
    project=str(RUN_DIR.parent),
    name=RUN_DIR.name,
    exist_ok=True,
    seed=42,
    plots=True,
)

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 10. 시험 학습 결과 확인 </h2>

<h3 style="color:#FFD866; border-left:3px solid #FFD866; padding-left:8px;"> 1. 결과 폴더와 생성 파일 확인 </h3>

In [ ]:
from IPython.display import display, Image
print('결과 폴더:', RUN_DIR)
print('폴더 존재:', RUN_DIR.exists())
if RUN_DIR.exists():
    for item in sorted(RUN_DIR.iterdir()):
        print(item.name)

<h3 style="color:#FFD866; border-left:3px solid #FFD866; padding-left:8px;"> 2. 학습 이미지와 정답 라벨 확인 </h3>

In [ ]:
for filename in ['train_batch0.jpg', 'train_batch1.jpg', 'train_batch2.jpg']:
    image_path = RUN_DIR / filename
    if image_path.exists():
        print(filename)
        display(Image(filename=str(image_path), width=1000))

<h3 style="color:#FFD866; border-left:3px solid #FFD866; padding-left:8px;"> 3. 검증 정답과 모델 예측 비교 </h3>

In [ ]:
for filename in ['val_batch0_labels.jpg', 'val_batch0_pred.jpg']:
    image_path = RUN_DIR / filename
    if image_path.exists():
        print(filename)
        display(Image(filename=str(image_path), width=1000))

<h3 style="color:#FFD866; border-left:3px solid #FFD866; padding-left:8px;"> 4. 학습 그래프 확인 </h3>

In [ ]:
results_path = RUN_DIR / 'results.png'
if results_path.exists():
    display(Image(filename=str(results_path), width=1100))
else:
    print('results.png을 찾을 수 없습니다:', results_path)

<h3 style="color:#FFD866; border-left:3px solid #FFD866; padding-left:8px;"> 5. 혼동행렬 확인 </h3>

In [ ]:
for filename in ['confusion_matrix.png', 'confusion_matrix_normalized.png']:
    image_path = RUN_DIR / filename
    if image_path.exists():
        print(filename)
        display(Image(filename=str(image_path), width=900))

<h3 style="color:#FFD866; border-left:3px solid #FFD866; padding-left:8px;"> 6. 성능 지표와 모델 파일 확인 </h3>

In [ ]:
BEST_MODEL = RUN_DIR / 'weights' / 'best.pt'
LAST_MODEL = RUN_DIR / 'weights' / 'last.pt'
print('best.pt:', BEST_MODEL.exists(), BEST_MODEL)
print('last.pt:', LAST_MODEL.exists(), LAST_MODEL)

if BEST_MODEL.exists():
    best_model = YOLO(str(BEST_MODEL))
    metrics = best_model.val(data=DATA_YAML, split='test', device=DEVICE, plots=True)
    print(f'Precision : {metrics.box.mp:.4f}')
    print(f'Recall    : {metrics.box.mr:.4f}')
    print(f'mAP50     : {metrics.box.map50:.4f}')
    print(f'mAP50-95  : {metrics.box.map:.4f}')

<h3 style="color:#FFD866; border-left:3px solid #FFD866; padding-left:8px;"> 7. Test 이미지 사람 탐지 결과 확인 </h3>

In [ ]:
test_images = sorted((Path(DATASET_PATH) / 'images' / 'test').glob('*'))
if not test_images:
    raise FileNotFoundError('Test 이미지를 찾을 수 없습니다.')
prediction_results = best_model.predict(
    source=str(test_images[0]), conf=0.5, save=True, project=str(RUN_DIR), name='prediction', exist_ok=True
)
result_image = RUN_DIR / 'prediction' / test_images[0].name
print('테스트 이미지:', test_images[0])
if result_image.exists():
    display(Image(filename=str(result_image), width=1000))

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 11. 본 학습 </h2>

In [ ]:
MAIN_RUN_DIR = Path('/content/drive/MyDrive/wardy/ml/src/export/person_detector_training')

main_model = YOLO('yolo11n.pt')
main_results = main_model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    device=DEVICE,
    workers=2,
    patience=10,
    project=str(MAIN_RUN_DIR.parent),
    name=MAIN_RUN_DIR.name,
    exist_ok=True,
    seed=42,
    plots=True,
)

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 12. 본 학습 결과 확인 </h2>

<h3 style="color:#FFD866; border-left:3px solid #FFD866; padding-left:8px;"> 1. 결과 폴더와 생성 파일 확인 </h3>

In [ ]:
print('본 학습 결과 폴더:', MAIN_RUN_DIR)
print('폴더 존재:', MAIN_RUN_DIR.exists())
if MAIN_RUN_DIR.exists():
    for item in sorted(MAIN_RUN_DIR.iterdir()):
        print(item.name)

<h3 style="color:#FFD866; border-left:3px solid #FFD866; padding-left:8px;"> 2. 학습 이미지와 정답 라벨 확인 </h3>

In [ ]:
for filename in ['train_batch0.jpg', 'train_batch1.jpg', 'train_batch2.jpg']:
    image_path = MAIN_RUN_DIR / filename
    if image_path.exists():
        print(filename)
        display(Image(filename=str(image_path), width=1000))

<h3 style="color:#FFD866; border-left:3px solid #FFD866; padding-left:8px;"> 3. 검증 정답과 모델 예측 비교 </h3>

In [ ]:
for filename in ['val_batch0_labels.jpg', 'val_batch0_pred.jpg']:
    image_path = MAIN_RUN_DIR / filename
    if image_path.exists():
        print(filename)
        display(Image(filename=str(image_path), width=1000))

<h3 style="color:#FFD866; border-left:3px solid #FFD866; padding-left:8px;"> 4. 학습 그래프와 혼동행렬 확인 </h3>

In [ ]:
for filename, width in [
    ('results.png', 1100),
    ('confusion_matrix.png', 900),
    ('confusion_matrix_normalized.png', 900),
]:
    image_path = MAIN_RUN_DIR / filename
    if image_path.exists():
        print(filename)
        display(Image(filename=str(image_path), width=width))

<h3 style="color:#FFD866; border-left:3px solid #FFD866; padding-left:8px;"> 5. Test 세트 성능 지표와 모델 파일 확인 </h3>

In [ ]:
MAIN_BEST_MODEL = MAIN_RUN_DIR / 'weights' / 'best.pt'
MAIN_LAST_MODEL = MAIN_RUN_DIR / 'weights' / 'last.pt'
print('best.pt:', MAIN_BEST_MODEL.exists(), MAIN_BEST_MODEL)
print('last.pt:', MAIN_LAST_MODEL.exists(), MAIN_LAST_MODEL)

if not MAIN_BEST_MODEL.exists():
    raise FileNotFoundError(f'본 학습 best.pt를 찾을 수 없습니다: {MAIN_BEST_MODEL}')

main_best_model = YOLO(str(MAIN_BEST_MODEL))
main_metrics = main_best_model.val(data=DATA_YAML, split='test', device=DEVICE, plots=True)
print(f'Precision : {main_metrics.box.mp:.4f}')
print(f'Recall    : {main_metrics.box.mr:.4f}')
print(f'mAP50     : {main_metrics.box.map50:.4f}')
print(f'mAP50-95  : {main_metrics.box.map:.4f}')

<h3 style="color:#FFD866; border-left:3px solid #FFD866; padding-left:8px;"> 6. Test 이미지 사람 탐지 결과 확인 </h3>

In [ ]:
main_test_images = sorted((Path(DATASET_PATH) / 'images' / 'test').glob('*'))
if not main_test_images:
    raise FileNotFoundError('Test 이미지를 찾을 수 없습니다.')

main_prediction_results = main_best_model.predict(
    source=str(main_test_images[0]),
    conf=0.5,
    save=True,
    project=str(MAIN_RUN_DIR),
    name='prediction',
    exist_ok=True,
)
main_result_image = MAIN_RUN_DIR / 'prediction' / main_test_images[0].name
print('테스트 이미지:', main_test_images[0])
if main_result_image.exists():
    display(Image(filename=str(main_result_image), width=1000))

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 13. ONNX 배포 모델 생성 </h2>

본 학습의 best.pt를 고정 입력 크기 640의 ONNX로 변환하고 Drive의 deploy 폴더에 보관합니다.

In [ ]:
import hashlib
import shutil

DEPLOY_DIR = MAIN_RUN_DIR / 'deploy'
DEPLOY_DIR.mkdir(parents=True, exist_ok=True)
ONNX_MODEL = DEPLOY_DIR / 'person_detector.onnx'

exported_onnx = Path(main_best_model.export(
    format='onnx',
    imgsz=640,
    batch=1,
    dynamic=False,
    simplify=True,
))
if exported_onnx.resolve() != ONNX_MODEL.resolve():
    shutil.copy2(exported_onnx, ONNX_MODEL)
if not ONNX_MODEL.is_file() or ONNX_MODEL.stat().st_size == 0:
    raise RuntimeError(f'ONNX 생성 실패: {ONNX_MODEL}')
onnx_sha256 = hashlib.sha256(ONNX_MODEL.read_bytes()).hexdigest()
print('ONNX:', ONNX_MODEL)
print(f'크기: {ONNX_MODEL.stat().st_size / 1024**2:.1f} MB')
print('SHA-256:', onnx_sha256)

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 14. PT와 ONNX Test 성능 비교 </h2>

ONNX 모델을 전체 test 세트에서 평가합니다. mAP50 하락이 0.02를 넘으면 배포를 중단합니다.

In [ ]:
onnx_model = YOLO(str(ONNX_MODEL))
onnx_metrics = onnx_model.val(
    data=DATA_YAML, split='test', imgsz=640, batch=1, device='cpu', plots=False
)
metric_pairs = {
    'Precision': (float(main_metrics.box.mp), float(onnx_metrics.box.mp)),
    'Recall': (float(main_metrics.box.mr), float(onnx_metrics.box.mr)),
    'mAP50': (float(main_metrics.box.map50), float(onnx_metrics.box.map50)),
    'mAP50-95': (float(main_metrics.box.map), float(onnx_metrics.box.map)),
}
print(f'{"Metric":<12} {"PT":>10} {"ONNX":>10} {"Delta":>10}')
for metric_name, (pt_value, onnx_value) in metric_pairs.items():
    print(f'{metric_name:<12} {pt_value:>10.4f} {onnx_value:>10.4f} {onnx_value - pt_value:>+10.4f}')
MAX_MAP50_DROP = 0.02
map50_drop = metric_pairs['mAP50'][0] - metric_pairs['mAP50'][1]
if map50_drop > MAX_MAP50_DROP:
    raise RuntimeError(f'ONNX mAP50이 PT보다 {map50_drop:.4f} 하락했습니다.')
print('ONNX 성능 검증 통과')

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 15. ONNX Test 이미지 탐지 확인 </h2>

In [ ]:
onnx_prediction_results = onnx_model.predict(
    source=str(main_test_images[0]), imgsz=640, conf=0.5, save=True,
    project=str(DEPLOY_DIR), name='onnx_prediction', exist_ok=True,
)
onnx_result_image = DEPLOY_DIR / 'onnx_prediction' / main_test_images[0].name
if onnx_result_image.exists():
    display(Image(filename=str(onnx_result_image), width=1000))

<h2 style="color:#FFD866; border-left:5px solid #FFD866; padding-left:12px;"> 16. 배포 파일과 Jetson 엔진 생성 </h2>

In [ ]:
summary_lines = [
    'Wardy Person Detector Deployment',
    f'PT model: {MAIN_BEST_MODEL}',
    f'ONNX model: {ONNX_MODEL}',
    f'ONNX SHA-256: {onnx_sha256}',
    'Input: 1x3x640x640',
    'Class 0: person',
    '',
]
for metric_name, (pt_value, onnx_value) in metric_pairs.items():
    summary_lines.append(
        f'{metric_name}: PT={pt_value:.4f}, ONNX={onnx_value:.4f}, delta={onnx_value - pt_value:+.4f}'
    )
DEPLOYMENT_SUMMARY = DEPLOY_DIR / 'deployment_summary.txt'
DEPLOYMENT_SUMMARY.write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')
print(DEPLOYMENT_SUMMARY.read_text(encoding='utf-8'))
print('Jetson으로 복사할 파일:', ONNX_MODEL)

### Jetson에서 TensorRT 엔진 생성

TensorRT 엔진은 Colab T4가 아니라 실제 사용할 Jetson에서 생성해야 합니다. 저장소와 person_detector.onnx를 Jetson으로 복사한 뒤 실행하세요.

<pre>chmod +x edge/scripts/build_person_detector_engine.sh
./edge/scripts/build_person_detector_engine.sh /path/to/person_detector.onnx /path/to/person_detector.engine</pre>

기존 엔진을 다시 만들 때만 마지막에 --force를 추가하세요. 스크립트는 Jetson 여부와 trtexec를 확인하고 FP16 엔진 생성 후 로드 벤치마크까지 수행합니다.